# EQO notebook: fault-tolerant memory study

This example combines **FTPrimitiveBench** circuit construction with **LightStim** logical-error estimation. The output Stim circuit becomes a typed workflow handoff; no simulator is imported into the notebook.

In [ ]:
import os

from eqo import EQOClient, render_artifact, render_run

eqo = EQOClient.connect(os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080"))
eqo.health()

## Run the published two-tool workflow

The workflow fixes a distance-three memory experiment and bounded sampling parameters suitable for a development demonstration. It is not a performance or threshold result.

In [ ]:
workflow = next((item for item in eqo.workflows.list() if item["id"] == "qec-memory-estimation"), None)
if workflow is None:
    raise RuntimeError("qec-memory-estimation is not published by this EQO profile.")
run = eqo.workflows.submit(
    workflow["id"], workflow["version"], execution_target="development-slurm-docker"
)
render_run(run)

In [ ]:
completed = run.wait(timeout=600)
if completed.state != "succeeded":
    raise RuntimeError(f"Memory study ended in {completed.state}; inspect render_run(completed).")
display(render_artifact(completed.artifacts.by_type("qhpc.stim-circuit@1")))
display(render_artifact(completed.artifacts.by_type("qhpc.logical-error-estimate@1")))

The run record and artifacts preserve the generator and estimator boundaries. To explore a multi-distance comparison, use the published `showcase-qec-distance-study` workflow through the Workbench, CLI, or the same SDK submission pattern.